# Combine Vpop designs and upload them to jinko

In [ ]:
# Jinko specifics imports & initialization
from jinko import JinkoClient

client = JinkoClient()
client.auth_check()

In [ ]:
# Cookbook specific imports
from crabbit import merge_vpop_designs

# Cookbook specific constants
folder_id = "965f821b-dc35-4cd8-a026-dd276ddcf536"

### Download the VpopDesigns you want to merge

In this use case, both vpop designs come from the same jinko.ai folder, fell free to change it

In [ ]:
# Check that we are in the correct folder and the desired items exist
# List all VpopDesign items in the specified folder
vpop_designs = list(client.iter_vpop_designs(folder=folder_id))

print("Available VPOP Designs in the folder:")
for design in vpop_designs:
    print(f"- {design.name}")

In [ ]:
# Name of the VpopDesign to download
vpop_design_name_to_download = ["vpop design parameters", "vpop design compartments"]

# it's the element of vpop_designs whose name is vpop_design_name_to_to_download
vpop_design_urls_to_merge = [
    vpop_design.url
    for vpop_design in vpop_designs
    if vpop_design.name in vpop_design_name_to_download
]

### Merge the vpop designs

In [ ]:
vpop_design_merged = merge_vpop_designs(vpop_design_urls_to_merge, client=client)

### Get the Ids of the combined model

In [ ]:
# List all ComputationalModel items in the specified folder
models = list(client.iter_models(folder=folder_id))

print("Available Models in the folder:")
for model in models:
    print(f"- {model.name}")

### Upload the combined VpopDesign to jinko with the attached model

In [ ]:
# Name of the model corresponding to the combined vpop
model_name = "simple tumor model"

# it's the element of models whose name is model_name
model = next((model for model in models if model.name == model_name), None)

# merge_vpop_designs drops the model link, so attach the model again
vpop_generator = {
    "contents": {
        **vpop_design_merged,
        "computationalModelId": {
            "coreItemId": model.core_id,
            "snapshotId": model.snapshot_id,
        },
    },
    "tag": "VpopGeneratorFromDesign",
}

vpop_design = client.create_vpop_design_from_json(
    json_content=vpop_generator,
    name="Combined Vpop Design",
    folder=folder_id,
)

print(f"Resource link: {vpop_design.url}")